## Miscellaneous Analysis Notebook

In [ ]:
import pandas as pd

# This cell is for reading in SPLASH's anchor annotations as a dataframe called 'splice'
# and reading the significant anchors output by supervised test as a dataframe called 'supervised'

splice = pd.read_csv("/projectnb/bf527/students/hbeakley/project/results/downstream_results/all_annotated_anchors.tsv", sep="\t")
supervised = pd.read_csv('/projectnb/bf527/students/hbeakley/project/results/GBM_supervised_metadata/GLM_supervised_anchors.tsv', sep="\t")

print(splice.shape)
print(supervised.shape)

print(supervised["anchor"].nunique())
print(splice["anchor"].nunique())

(1455978, 35)
(230, 11)
230
425198


In [7]:
# Here, I merge the splice and supervised dataframes

common_rows = pd.merge(splice, supervised, how='inner', on='anchor')

print(common_rows.shape)
print(common_rows["anchor"].nunique())

common_rows.to_csv("sig_spliced_anchors.tsv", sep='\t', index=False)

Index(['extendor_index', 'anchor', 'target', 'target_count', 'pval_opt',
       'effect_size_bin', 'anchor_count', 'anch_uniqTargs',
       'number_nonzero_samples', 'target_entropy',
       'avg_hamming_distance_max_target', 'avg_hamming_distance_all_pairs',
       'avg_edit_distance_max_target', 'avg_edit_distance_all_pairs',
       'pval_opt_corrected', 'target_fraction', 'num_targets_per_anchor',
       'extendor', 'extendor_order', 'anchor_index', 'STAR_flag', 'STAR_chr',
       'STAR_coord', 'STAR_CIGAR', 'STAR_num_alignments',
       'STAR_num_mismatches', 'is.aligned_STAR', 'is.STAR_chimeric',
       'is.STAR_SJ', 'is.aligned_Bowtie', 'extendor_gene',
       'num_extendor_gene_anchor', 'all_splice_juncs', 'all_SS_AS_annot',
       'all_SS_annot'],
      dtype='object')
Index(['anchor', 'effect_size_bin', 'number_nonzero_samples',
       'most_freq_target_1', 'cnt_most_freq_target_1', 'most_freq_target_2',
       'cnt_most_freq_target_2', 'avg_hamming_distance_max_target',
     

In [4]:
# Here, I read in a different collection of supervised anchors based on a slightly modified supervised test script
expanded_supervised = pd.read_csv('/projectnb/bf527/students/hbeakley/project/results/GBM_robustify_3.0_supervised_metadata/GLM_supervised_anchors.tsv', sep="\t")

# merged expanded_supervised dataframe with the original splice dataframe
common_rows_2 = pd.merge(splice, expanded_supervised, how='inner', on='anchor')

print(expanded_supervised.shape)
print(common_rows_2.shape)
print(common_rows_2["anchor"].nunique())

# write the merged dataframe to an output file
common_rows_2.to_csv("annotated_supervised_3.tsv", sep='\t', index=False)

# generate a clean set of genes from the merged dataframe to use for GO analysis
genes = set(common_rows_2["extendor_gene"].tolist())
genes = {g for g in genes if pd.notna(g)}
clean_genes = set()
output_file = "clean_genes_3.0.txt"

for line in genes:
    # split on commas
    parts = line.strip().split(",")
    for gene in parts:
        gene = gene.strip()
        if gene == "":
            continue
        
        # OPTIONAL: remove LOC IDs
        if gene.startswith("LOC"):
            continue
        
        clean_genes.add(gene)

# Write clean list
with open(output_file, "w") as out:
    for gene in sorted(clean_genes):
        out.write(gene + "\n")

print(f"Done! Wrote {len(clean_genes)} cleaned gene symbols to {output_file}")

(220, 11)
(609, 45)
202
Done! Wrote 454 cleaned gene symbols to clean_genes_3.0.txt


In [5]:
# curious about how many anchors overlap between the original and altered supervised analyses
supervised_method_overlap = expanded_supervised[expanded_supervised["anchor"].isin(supervised["anchor"].tolist())]
print(supervised_method_overlap.shape)

(77, 11)


In [8]:
# Here, I isolated the significantly alternatively spliced genes from Wang et al.'s results

import pandas as pd
import numpy as np

# path to your downloaded S7 file
xlsx_path = "/projectnb/bf527/students/hbeakley/project/inputs/diff_splicers_wang_data.xlsx"   # change to your actual filename

df = pd.read_excel(xlsx_path)

# sanity check: see the column names
delta_col = "E(dPSI) per LSV junction"
prob_col  = "P(|dPSI|>=0.10) per LSV junction"
lsv_type_col = "LSV Type"

def event_is_significant(row):
    # Split semicolon-separated values
    deltas = np.array([float(x) for x in str(row[delta_col]).split(";")])
    probs  = np.array([float(x) for x in str(row[prob_col]).split(";")])

    # Rule 1: AS-type events only
    # Valid tags: A5SS, A3SS, ES, and 's' or 't' splice LSVs
    lsv_type = str(row[lsv_type_col])
    if not any(tag in lsv_type for tag in ["A5SS", "A3SS", "ES", "s", "t"]):
        return False

    # Rule 2: any junction with |ΔPSI| >= 0.10
    mask_dpsi = np.abs(deltas) >= 0.10

    # Rule 3: corresponding probability >= 0.75
    mask_prob = probs >= 0.75

    return np.any(mask_dpsi & mask_prob)

# Apply filter
sig_mask = df.apply(event_is_significant, axis=1)
full_set = df[sig_mask].copy()

print("Rows:", full_set.shape[0])
print("Unique genes:", full_set["Gene Name"].nunique())

Rows: 443
Unique genes: 299


In [13]:
# how many of my genes overlap with theirs? Spoiler alert: not that many

wang_genes = set(full_set["Gene Name"].tolist())
intersection = wang_genes & clean_genes

print(intersection)

print(len(intersection))
print(len(wang_genes))
print(len(clean_genes))

{'PDE6B', 'RAPH1', 'RTN4', 'NPIPB3', 'USP54', 'NPIPA2', 'SORBS1', 'ZNF254', 'KIF1B', 'ZNF561', 'GRB10', 'GSN', 'R3HDM2', 'NAV1'}
14
299
454
